# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and their `@id`s. This helps us identify how to load and reference records and columns in a semantic, robust manner.

In [ ]:
# List all record sets and their fields using @id
record_sets = []
try:
    # Iterate over record sets
    for rs in getattr(metadata, "record_sets", []):
        print(f"Record Set: name='{rs.name}', @id='{rs.id}'")
        record_sets.append(rs.id)
        if hasattr(rs, 'fields'):
            for field in rs.fields:
                print(f"    Field: name='{field.name}', @id='{field.id}', dataType={getattr(field, 'data_type', None)}")
except AttributeError:
    # Fallback if not exposed via metadata.record_sets (older croissant spec)
    try:
        # Try extracting the list via to_json
        meta_json = metadata.to_json() if hasattr(metadata, 'to_json') else metadata
        record_sets_json = meta_json.get('recordSet', [])
        for rs in record_sets_json:
            rs_id = rs.get('@id')
            print(f"Record Set @id: {rs_id}")
            record_sets.append(rs_id)
            if 'field' in rs:
                for field in rs['field']:
                    fname = field.get('name')
                    fid = field.get('@id')
                    dtype = field.get('dataType')
                    print(f"    Field: name='{fname}', @id='{fid}', dataType={dtype}")
    except Exception as e:
        print('Could not extract record sets from metadata:', e)

if len(record_sets) == 0:
    # Try fetching all available record_set ids
    print("No record sets explicitly listed in metadata. Attempting to guess record_set ids from the dataset...")
    # Try using 'dataset._record_set_names' (undocumented, may not be present)
    if hasattr(dataset, '_record_set_names'):
        record_sets = list(dataset._record_set_names)
        print('Detected record sets:', record_sets)
    else:
        record_sets = []
if len(record_sets) == 0:
    # Manual fallback based on mlcroissant convention
    record_sets = ['cr:RecordSet_1']  # Placeholder - update with actual record set ids if known
    print("Manually set a placeholder RecordSet (please adjust with actual @id from data overview)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set (@id)
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for record set @id={record_set_id}")
    try:
        recs = list(dataset.records(record_set=record_set_id))
        if len(recs) > 0:
            df = pd.DataFrame(recs)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set {record_set_id}.")
            print("Fields (columns) detected:", list(df.columns))
            display(df.head())
        else:
            print(f"No records found for record set {record_set_id}.")
    except Exception as e:
        print(f"Error loading records for record set {record_set_id}: {e}")

# For further analysis, choose the first successfully loaded record set
if len(dataframes) > 0:
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"Proceeding with record set: {selected_record_set_id}")
    df = dataframes[selected_record_set_id]
else:
    raise RuntimeError('No record sets could be loaded. Please check the record set @id references and dataset structure.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates filtering, normalization, and group analysis. **All fields and columns are referenced by their `@id` or dataset-extracted names.**

In [ ]:
# List all columns
print('Fields (columns) available:', df.columns.tolist())

# Select a numeric field for analysis
# To choose a field, list all numeric-like columns
numeric_candidates = df.select_dtypes(include=np.number).columns.tolist()
if not numeric_candidates:
    # Try to guess numeric columns (sometimes ML Croissant returns columns as object)
    for col in df.columns:
        # Check if all values can be cast to float
        try:
            df[col] = pd.to_numeric(df[col])
            numeric_candidates.append(col)
        except Exception:
            pass
    numeric_candidates = list(set(numeric_candidates))
print('Numeric field candidates:', numeric_candidates)

if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Selected numeric field for analysis: {numeric_field}")
else:
    raise ValueError('No numeric field found for EDA.')

# Example filtering: keep records greater than threshold
threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
display(filtered_df.head())

# Normalize the numeric field (z-score normalization)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Pick a categorical field for grouping (if any)
categorical_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
if len(categorical_candidates) > 0:
    group_field = categorical_candidates[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
    print(f"Grouped mean {numeric_field} by {group_field} (top 5):")
    display(grouped_df.head())
else:
    print("No categorical fields available for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Plot the distribution of the selected numeric field
plt.figure(figsize=(8, 5))
plt.hist(df[numeric_field].dropna(), bins=30, color='skyblue', edgecolor='black')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.title(f'Distribution of {numeric_field}')
plt.show()

# If we did grouping, show as a barplot
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10, 5))
    plt.bar(grouped_df[group_field][:10], grouped_df[numeric_field][:10], color='salmon', edgecolor='black')
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.title(f'Top 10 {group_field} by mean {numeric_field}')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrates how to use the `mlcroissant` library to access and analyze the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset via its Croissant schema.

- Record sets, fields, and columns are referenced by their `@id` fields as provided in the metadata.
- Data is programmatically extracted and analyzed using Pandas and visualized with matplotlib.
- Use similar workflows for additional advanced analyses or machine learning tasks as appropriate.